In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, confusion_matrix
from xgboost import XGBClassifier





df = pd.read_csv("model/featured_transactions.csv")

X = df.drop(columns=[
    "customer_id",                 
    "transaction_number",          
    "hour",                         
    "day_of_week",                  
    "weekend_flag",                 
    "night_flag"                    
])

y = df["fraud_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



model = XGBClassifier(

    n_estimators=300,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),

    random_state=42,

    eval_metric="logloss",

    n_jobs=-1
)


model.fit(X_train, y_train)

y_pred = model.predict(X_test)


print("=" * 60)
print("Classification Report")
print("=" * 60)

print(classification_report(y_test, y_pred))

print("=" * 60)

print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred):.4f}")

print("=" * 60)

print("Confusion Matrix")

print(confusion_matrix(y_test, y_pred))

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19600
           1       0.89      0.79      0.84       400

    accuracy                           0.99     20000
   macro avg       0.94      0.90      0.92     20000
weighted avg       0.99      0.99      0.99     20000

Precision : 0.8930
Recall    : 0.7925
F1 Score  : 0.8397
Confusion Matrix
[[19562    38]
 [   83   317]]


In [ ]:
feature_importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": model.feature_importances_

})
feature_importance = feature_importance.sort_values(

    by="Importance",

    ascending=False

)

print(feature_importance)

                        Feature  Importance
2                  city_changed    0.298503
8           amount_growth_ratio    0.285193
1                device_changed    0.118800
7              amount_deviation    0.076075
5           customer_avg_amount    0.039958
9                 amount_vs_max    0.034448
6           customer_max_amount    0.033979
4   previous_transaction_amount    0.032620
0                        amount    0.030812
10   transaction_count_last_24h    0.028170
3        payment_method_changed    0.021442


In [7]:
numerical = [
    "amount",
    "customer_avg_amount",
    "customer_max_amount",
    "previous_transaction_amount",
    "amount_deviation",
    "amount_growth_ratio",
    "amount_vs_max",
    "transaction_count_last_24h"
]

print(df[numerical].corr())

                               amount  customer_avg_amount  \
amount                       1.000000             0.699002   
customer_avg_amount          0.699002             1.000000   
customer_max_amount          0.552163             0.810429   
previous_transaction_amount  0.533786             0.731436   
amount_deviation             0.679012            -0.050358   
amount_growth_ratio          0.214474            -0.019820   
amount_vs_max                0.141198            -0.027859   
transaction_count_last_24h   0.016224             0.021948   

                             customer_max_amount  previous_transaction_amount  \
amount                                  0.552163                     0.533786   
customer_avg_amount                     0.810429                     0.731436   
customer_max_amount                     1.000000                     0.597212   
previous_transaction_amount             0.597212                     1.000000   
amount_deviation                    

In [ ]:
import joblib

joblib.dump(model, "model/fraud_detection_model.pkl")

print("Model saved successfully.")

Model saved successfully.
